In [5]:
# --no-check-certificate 这个参数是核武器，强行无视任何证书拦截！
!wget -q --show-progress --no-check-certificate https://hf-mirror.com/timm/tf_efficientnet_b3.ns_jft_in1k/resolve/main/model.safetensors -O effb3_ns.safetensors

import os
# 增加一个体检机制，帮你确认有没有被镜像站坑了
size_mb = os.path.getsize('effb3_ns.safetensors') / (1024 * 1024)
print(f"📦 检查文件大小: {size_mb:.2f} MB (如果只有零点几MB，说明下载失败被拦截了！)")
if size_mb > 40:
    print("✅ 预训练权重下载真正完成！")
else:
    print("🚨 警告：下载下来的可能是个假文件（网页报错），建议换个时间或用浏览器下载传上去！")

effb3_ns.safetensor 100%[===================>]  47.05M  5.08MB/s    in 10s     
📦 检查文件大小: 47.05 MB (如果只有零点几MB，说明下载失败被拦截了！)
✅ 预训练权重下载真正完成！


In [1]:
import os
import gc
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import log_loss, confusion_matrix
import timm
from transformers import get_cosine_schedule_with_warmup

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# ==========================================
# ⚙️ 1. 全局配置参数 (RTX 5090 极速微调版)
# ==========================================
CSV_PATH = 'dataset/driver_imgs_list.csv'

MODEL_NAME = 'tf_efficientnet_b3.ns_jft_in1k'
IMG_SIZE = 300      # EfficientNet-B3 原生分辨率
EPOCHS = 3          # ✅ 伪标签微调只需要 3 轮
BATCH_SIZE = 64     
ACCUMULATION_STEPS = 1  
NUM_SPLITS = 5
TARGET_FOLDS = [0, 1, 2, 3, 4]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 开启 TF32 加速
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 📊 2. 数据划分 (GroupKFold 隔离策略)
# ==========================================
def generate_balanced_folds(csv_path, n_splits=5):
    df = pd.read_csv(csv_path).reset_index(drop=True)
    if 'label_int' not in df.columns:
        df['label_int'] = df['classname'].str.extract(r'(\d+)').astype(int)

    driver_counts = df.groupby('subject').size().sort_values(ascending=False)
    fold_totals = np.zeros(n_splits)
    fold_groups = [[] for _ in range(n_splits)]

    for subject, count in driver_counts.items():
        min_fold_idx = np.argmin(fold_totals)
        fold_groups[min_fold_idx].append(subject)
        fold_totals[min_fold_idx] += count

    df['fold'] = -1
    for i, subjects in enumerate(fold_groups):
        df.loc[df['subject'].isin(subjects), 'fold'] = i

    print("\n" + "="*50)
    print("            📊 5 折原始数据分布情况")
    print("="*50)
    for i in range(n_splits):
        print(f"Fold {i}  |  驾驶员数量: {len(fold_groups[i]):2d}  |  图片总数: {int(fold_totals[i]):4d}")
    print("="*50 + "\n")
    return df

# ==========================================
# 🖼️ 3. 数据集与增强
# ==========================================
def get_train_transforms(img_size=300):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.6),
        A.Affine(translate_percent=(-0.05, 0.05), scale=(0.95, 1.05), rotate=(-10, 10), p=0.5),
        A.GaussNoise(p=0.4),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

def get_valid_transforms(img_size=300):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

class DriverDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 👑 智能路由：判断图片来源于原始训练集还是伪标签测试集
        if row['subject'] == 'pseudo_test':
            img_path = os.path.join('./dataset/imgs/test_cropped_v2', row['img'])
        else:
            img_path = os.path.join('./dataset/imgs/train_cropped_v2', row['classname'], row['img'])
        
        image = cv2.imread(img_path)
        
        # 防御性回退机制
        if image is None:
            if row['subject'] == 'pseudo_test':
                image = cv2.imread(os.path.join('./dataset/imgs/test', row['img']))
            else:
                image = cv2.imread(os.path.join('./dataset/imgs/train', row['classname'], row['img']))
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, row['label_int']

# ==========================================
# 🛠️ 4. 评估工具
# ==========================================
def plot_confusion_matrix(y_true, y_pred, fold_idx, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues')
    plt.title(f'Fold {fold_idx} Confusion Matrix (Pseudo)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(save_dir, f'pseudo_cm_fold_{fold_idx}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def generate_grad_cam(model, val_loader, fold_idx, save_dir):
    model.eval()
    try:
        images, labels = next(iter(val_loader))
    except StopIteration:
        return

    target_layer = model.conv_head 
    cam = GradCAM(model=model, target_layers=[target_layer])
    input_tensor = images[0:1].to(DEVICE)

    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    img_np = images[0].permute(1, 2, 0).cpu().numpy()

    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img_np = np.clip(std * img_np + mean, 0, 1)

    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    plt.imsave(os.path.join(save_dir, f"pseudo_grad_cam_fold_{fold_idx}.png"), cam_image)

# ==========================================
# 🚀 5. 主干微调流程
# ==========================================
def main():
    base_dir = 'models/effb3' 
    os.makedirs(base_dir, exist_ok=True)

    full_df = generate_balanced_folds(CSV_PATH, NUM_SPLITS)
    oof_preds = np.zeros((len(full_df), 10))

    # 👑 加载提纯后的伪标签数据
    pseudo_df = pd.read_csv('pseudo_labels.csv')

    for fold in TARGET_FOLDS:
        print(f"\n{'='*40}\n🌟 开始微调 EffB3 Fold {fold} (带伪标签加持) 🌟\n{'='*40}")

        # 验证集保持绝对纯净，不能有伪标签穿越！
        val_df = full_df[full_df['fold'] == fold]
        
        # 训练集混合伪标签
        original_train = full_df[full_df['fold'] != fold]
        train_df = pd.concat([original_train, pseudo_df], axis=0).reset_index(drop=True)
        
        print(f"  -> 当前折训练集数量: {len(train_df)} (包含 {len(pseudo_df)} 张伪标签)")
        print(f"  -> 当前折验证集数量: {len(val_df)}")

        # ✅ 启用了去掉 img_dir 的智能路由版 DriverDataset
        train_loader = DataLoader(
            DriverDataset(train_df, transform=get_train_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True, drop_last=True
        )
        val_loader = DataLoader(
            DriverDataset(val_df, transform=get_valid_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True
        )

        # 1. 建立空模型
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10, drop_path_rate=0.2)
        
        # 2. 👑 加载上一轮最优秀的模型作为起点
        previous_best_weight = os.path.join(base_dir, f"best_model_effb3_fold_{fold}.pth")
        print(f"📥 正在继承上一轮优秀的 EffB3 权重: {previous_best_weight}")
        model.load_state_dict(torch.load(previous_best_weight, map_location=DEVICE, weights_only=True))
        model.to(DEVICE)

        class_weights = torch.tensor([1.0]*9 + [1.5], dtype=torch.float).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        
        # ✅ CNN 微调学习率降低 (原为 3e-4，微调使用 3e-5)
        optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
        scaler = torch.amp.GradScaler('cuda')

        total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_steps * 0.1),
            num_training_steps=total_steps
        )

        best_val_loss = float('inf')
        # ✅ 保存名字加上 pseudo_ 前缀
        save_path = os.path.join(base_dir, f"pseudo_best_model_effb3_fold_{fold}.pth")
        EARLY_STOP_PATIENCE = 2
        epochs_no_improve = 0

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
            for i, (images, labels) in enumerate(pbar):
                images, labels = images.to(DEVICE), labels.to(DEVICE)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels) / ACCUMULATION_STEPS

                scaler.scale(loss).backward()
                train_loss += loss.item() * ACCUMULATION_STEPS

                if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                pbar.set_postfix({'loss': f"{loss.item()*ACCUMULATION_STEPS:.4f}"})

            model.eval()
            val_loss = 0.0
            fold_preds_list, fold_labels = [], []

            vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
            with torch.no_grad():
                for images, labels in vbar:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    fold_preds_list.append(outputs.softmax(dim=1).cpu().numpy())
                    fold_labels.extend(labels.cpu().numpy())
                    vbar.set_postfix({'v_loss': f"{loss.item():.4f}"})

            avg_val_loss = val_loss / len(val_loader)
            current_fold_preds = np.concatenate(fold_preds_list, axis=0)

            if avg_val_loss < best_val_loss:
                print(f"✅ 验证集 Loss 降低: {best_val_loss:.4f} -> {avg_val_loss:.4f}，保存微调模型。")
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), save_path)
                epochs_no_improve = 0 
                
                oof_preds[val_df.index] = current_fold_preds
                best_labels = fold_labels
                best_preds = np.argmax(current_fold_preds, axis=1)
            else:
                epochs_no_improve += 1
                print(f"⚠️ 验证集 Loss 未降低 ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f"🛑 连续 {EARLY_STOP_PATIENCE} 轮未下降，触发早停！")
                    break

        print("📸 正在生成混淆矩阵与 Grad-CAM 注意力热力图...")
        plot_confusion_matrix(best_labels, best_preds, fold, base_dir)
        
        model.load_state_dict(torch.load(save_path))
        generate_grad_cam(model, val_loader, fold, base_dir)

        del model, optimizer, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    print("\n🎉 所有 Fold 微调完毕！保存 OOF 预测结果...")
    
    # ✅ 加上 pseudo_ 前缀保存 OOF
    np.save(os.path.join(base_dir, "pseudo_oof_preds_effb3.npy"), oof_preds)

    final_labels = full_df['label_int'].values
    final_log_loss = log_loss(final_labels, oof_preds)
    print(f"🔥 伪标签加持后 Final OOF Log Loss (EfficientNet-B3): {final_log_loss:.4f}")

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info The read operation timed out
  data = fetch_version_info()



            📊 5 折原始数据分布情况
Fold 0  |  驾驶员数量:  5  |  图片总数: 4407
Fold 1  |  驾驶员数量:  5  |  图片总数: 4475
Fold 2  |  驾驶员数量:  5  |  图片总数: 4364
Fold 3  |  驾驶员数量:  6  |  图片总数: 4704
Fold 4  |  驾驶员数量:  5  |  图片总数: 4474


🌟 开始微调 EffB3 Fold 0 (带伪标签加持) 🌟
  -> 当前折训练集数量: 75645 (包含 57628 张伪标签)
  -> 当前折验证集数量: 4407
📥 正在继承上一轮优秀的 EffB3 权重: models/effb3/best_model_effb3_fold_0.pth


AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [2]:
import os
import torch
import pandas as pd
import numpy as np
import timm
from tqdm.notebook import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# ⚙️ 1. 基础配置 (RTX 5090 极速版)
# ==========================================
# 🔥 5090 专属加速魔法：开启 TF32 核心
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

FOLDS = [0, 1, 2, 3, 4]
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
TEST_DIR = 'dataset/imgs/test_cropped_v2'  # 确保这里读取的是你裁剪后的测试集

# ✅ 路径与模型完全对齐训练脚本
SAVE_DIR = './models/effb3'
MODEL_NAME = 'tf_efficientnet_b3.ns_jft_in1k'
WEIGHT_PATH_TEMPLATE = os.path.join(SAVE_DIR, 'pseudo_best_model_effb3_fold_{}.pth')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 300      # 必须和训练时保持绝对一致
BATCH_SIZE = 128    # 5090 显存极大，纯推理可以开到 128 甚至 256 起飞
NUM_WORKERS = 8

# ==========================================
# 🖼️ 2. 构建测试集 DataLoader (按 CSV 严格保序)
# ==========================================
class TestDriverDataset(Dataset):
    def __init__(self, csv_path, test_dir):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"🚨 找不到官方提交模板: {csv_path}")
        self.df = pd.read_csv(csv_path)
        self.test_dir = test_dir
        
        self.image_names = self.df['img'].values

        # ✅ 测试集只需要基础缩放和归一化，绝对不能加随机增强
        self.transform = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.test_dir, img_name)
        
        image = cv2.imread(img_path)
        if image is None:
            # 防御性回退：如果裁剪文件夹里没有，去原图文件夹找
            fallback_path = os.path.join('./dataset/imgs/test', img_name)
            image = cv2.imread(fallback_path)
            if image is None:
                raise FileNotFoundError(f"找不到图片: {img_path}")
                
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_tensor = self.transform(image=image)['image']

        return image_tensor, img_name

# ==========================================
# 🚀 3. 核心预测与 5折融合 (Ensemble)
# ==========================================
def generate_submission():
    print(f"🚀 启动预测流程！使用设备: {DEVICE}")

    test_dataset = TestDriverDataset(SAMPLE_SUBMISSION_PATH, TEST_DIR)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"📦 严格按照 CSV 顺序，共抓取 {len(test_dataset)} 张测试图片。")

    all_fold_preds = []

    for fold in FOLDS:
        weight_path = WEIGHT_PATH_TEMPLATE.format(fold)
        print(f"\n👨‍⚖️ 正在准备第 {fold} 号模型...")

        if not os.path.exists(weight_path):
            print(f"⚠️ 找不到权重文件 {weight_path}，跳过此折！")
            continue

        # 1. 实例化干净的空模型 (分类头必须直接设定为 10)
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10)
        model.to(DEVICE)
        
        # 2. 直接加载我们在训练阶段保存的完美权重
        state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state_dict, strict=True)
        model.eval()

        # 3. 编译加速 (如果 PyTorch 版本 >= 2.0，强烈建议开启)
        if int(torch.__version__.split('.')[0]) >= 2:
            model = torch.compile(model)

        fold_preds = []
        with torch.no_grad():
             # 使用 AMP 半精度推理，速度翻倍
             with torch.amp.autocast('cuda'):
                pbar = tqdm(test_loader, desc=f"⚡ Fold {fold} 推理中")
                for images, _ in pbar:
                    images = images.to(DEVICE)
                    outputs = model(images)
                    probs = torch.softmax(outputs, dim=1)
                    fold_preds.append(probs.cpu().numpy())

        fold_preds = np.concatenate(fold_preds, axis=0)
        all_fold_preds.append(fold_preds)
        
        # 释放显存
        del model
        torch.cuda.empty_cache()

    if not all_fold_preds:
        print("❌ 没有找到任何有效的权重文件，推理终止。")
        return

    # ==========================================
    # 🤝 4. 终极平均融合生成提交文件
    # ==========================================
    print(f"\n🤝 正在对找到的 {len(all_fold_preds)} 个模型进行概率平均融合...")
    all_fold_preds = np.array(all_fold_preds) 
    
    # 简单算术平均法 (Simple Average Ensemble)
    final_preds = np.mean(all_fold_preds, axis=0) 

    print("📝 正在生成最终的 CSV 提交文件...")
    img_filenames = test_dataset.image_names

    df_submit = pd.DataFrame(final_preds, columns=[f'c{i}' for i in range(10)])
    df_submit.insert(0, 'img', img_filenames) 

    submit_filename = os.path.join(SAVE_DIR, 'submission_effb3_5fold.csv')
    df_submit.to_csv(submit_filename, index=False)
    print(f"🎉 大功告成！预测文件已安全保存为: {submit_filename}")

if __name__ == '__main__':
    generate_submission()

🚀 启动预测流程！使用设备: cuda
📦 严格按照 CSV 顺序，共抓取 79726 张测试图片。

👨‍⚖️ 正在准备第 0 号模型...


⚡ Fold 0 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 1 号模型...


⚡ Fold 1 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 2 号模型...


...


👨‍⚖️ 正在准备第 3 号模型...


⚡ Fold 3 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 4 号模型...


⚡ Fold 4 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


🤝 正在对找到的 5 个模型进行概率平均融合...
📝 正在生成最终的 CSV 提交文件...
🎉 大功告成！预测文件已安全保存为: ./models/effb3/submission_effb3_5fold.csv


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import timm
from tqdm.notebook import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_DIR = 'dataset/imgs/test_cropped_v2'
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
IMG_SIZE = 300  # 👈 EffB3 专属尺寸
BATCH_SIZE = 128
FOLDS = [0, 1, 2, 3, 4]

# 加载测试集路径
df_test = pd.read_csv(SAMPLE_SUBMISSION_PATH)
image_names = df_test['img'].values

# 标准 ImageNet 归一化
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class TestFeatureDataset(Dataset):
    def __len__(self): return len(image_names)
    def __getitem__(self, idx):
        img_path = os.path.join(TEST_DIR, image_names[idx])
        image = cv2.imread(img_path)
        if image is None:
            image = cv2.imread(os.path.join('./dataset/imgs/test', image_names[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return transform(image=image)['image']

print("🚀 准备提取 EfficientNet-B3 5折骨干特征...")
test_loader = DataLoader(TestFeatureDataset(), batch_size=BATCH_SIZE, shuffle=False, num_workers=8)

all_fold_features = []

for fold in FOLDS:
    print(f"\n👨‍⚖️ 正在加载 EffB3 Fold {fold} 模型...")
    
    # 1. 每次循环创建一个干净的模型
    model = timm.create_model('tf_efficientnet_b3.ns_jft_in1k', pretrained=False, num_classes=10)
    
    # 2. 读取对应折的权重
    weight_path = f'pseudo_models/effb3/best_model_effb3_fold_{fold}.pth'
    if not os.path.exists(weight_path):
        print(f"⚠️ 找不到权重 {weight_path}，跳过此折！")
        continue
        
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE, weights_only=True))

    # 3. 切除分类头！对于 CNN，这会暴露出 Global Average Pooling 后的向量
    model.reset_classifier(0)
    model.to(DEVICE)
    model.eval()

    # 编译加速
    if int(torch.__version__.split('.')[0]) >= 2:
        model = torch.compile(model)

    fold_features = []
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            for images in tqdm(test_loader, desc=f"💎 提取 Fold {fold} 特征"):
                features = model(images.to(DEVICE))
                fold_features.append(features.cpu().numpy())

    current_fold_final_features = np.concatenate(fold_features, axis=0)
    all_fold_features.append(current_fold_final_features)
    
    del model
    torch.cuda.empty_cache()

if not all_fold_features:
    raise ValueError("🚨 没有成功提取到任何特征！")

# 🎯 对 5 折特征求算术平均
print("\n🤝 正在对 5 折特征进行算术平均融合...")
all_fold_features = np.array(all_fold_features) 
final_avg_features = np.mean(all_fold_features, axis=0) 

save_path = 'models/test_features_effb3.npy'
np.save(save_path, final_avg_features)
print(f"✅ EffB3 5折特征融合完毕！矩阵形状: {final_avg_features.shape}")
print(f"📁 已安全保存至: {save_path}")

🚀 准备提取 EfficientNet-B3 5折骨干特征...

👨‍⚖️ 正在加载 EffB3 Fold 0 模型...


💎 提取 Fold 0 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 EffB3 Fold 1 模型...


💎 提取 Fold 1 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 EffB3 Fold 2 模型...


...


👨‍⚖️ 正在加载 EffB3 Fold 3 模型...


💎 提取 Fold 3 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 EffB3 Fold 4 模型...


💎 提取 Fold 4 特征:   0%|          | 0/623 [00:00<?, ?it/s]


🤝 正在对 5 折特征进行算术平均融合...
✅ EffB3 5折特征融合完毕！矩阵形状: (79726, 1536)
📁 已安全保存至: models/test_features_effb3.npy
